# Day 1 · 2 · 3 — 같은 일을 흐름으로 비교

앞부분은 Day 1 vs Day 3 흐름 비교다. 파일 아래쪽에 오늘 수업 노트북 네 개를 원문 그대로 붙였다.

- `01_response_api.ipynb`
- `02_stream.ipynb`
- `03_ops.ipynb`
- `04_other_models.ipynb`

위에서 아래로 실행한다. `temperature` 는 오늘 모델에서 400 이다.


## 0. 키를 환경에서 읽고, 비교용 모델을 연다

`os.getenv`는 `.env` 파일을 열지 않는다. 파일이 있으면 직접 읽어서 `os.environ`에 넣은 다음, `OpenAI()`가 그 값을 찾는다.

수업에서 Day 1·2는 `gpt-5.4-nano`, Day 3는 `gpt-5.6-luna`를 썼다. 이 노트북은 **API 차이가 보이게** 한 모델로 맞춘다. `temperature`는 넣지 않는다 — 오늘 모델은 400이다.


In [ ]:
import json
import os
import time
import uuid
import urllib.request
from pathlib import Path

from openai import OpenAI
from pydantic import BaseModel

HERE = Path.cwd().resolve()
if HERE.name == "day03":
    ROOT = HERE.parent
elif (HERE / "llm-api-playground").is_dir():
    ROOT = HERE / "llm-api-playground"
else:
    ROOT = HERE

for env_path in (ROOT / ".env", ROOT / "env"):
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if "=" in line and "key" in line.lower() and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

key = os.getenv("OPENAI_API_KEY")
print("cwd :", Path.cwd())
print("키  :", (key[:8] + "…") if key else "없음 — env/.env 확인")

client = OpenAI()
API_MODEL = "gpt-5.6-luna"
print("모델:", API_MODEL)
print("문서: https://developers.openai.com/api/reference/resources/responses/methods/create")


## 삼일을 한 그림으로

```
Day 1  나 → messages[] 통째로 → chat.completions.create
                              ← choices[0].message.content
         다음 턴: 배열에 append 하고 처음부터 다시 부친다

Day 2  나 → messages[] + tools 설명서 → chat.completions.create
                              ← tool_calls (요청서) 또는 content (말)
         내가 함수 실행
         나 → messages[] + role:tool → create 다시
                              ← content (말)

Day 3  나 → input 한 줄 (+ 필요하면 tools) → responses.create
                              ← output[] 아이템들 + output_text
         다음 턴: previous_response_id 만 보낸다
         내 함수면 function_call_output 조각 하나
         내장이면 왕복 루프 자체가 내 코드에 없다
```

세 답이 갈리는 곳:

| 무엇이 | Day 1·2 | Day 3 |
|---|---|---|
| 배열 재전송 | 내 프로그램 | 서버 (`previous_response_id`) |
| 도구 왕복 구조 | 내 코드에 있다 | **그대로** 내 코드에 있다 |
| 도구 실행 (내장) | — | OpenAI 서버 (토큰으로 청구) |


---

## 흐름 1. 첫 호출 — 답이 상자 어디에 있나

같은 질문 한 줄. 다른 문은 `create`의 이름과, 답을 꺼내는 경로다.

```
질문 "한 문장으로 자기소개."
        │
        ├─ Day 1  messages=[{role, content}]  →  r.choices[0].message.content
        └─ Day 3  input="…"                   →  r.output_text
```

`r = ...` 는 답을 저장하는 게 아니다. 서버가 돌려준 **상자 전체**에 이름을 붙이는 것이다. 안 붙이면 상자에서 답·id·토큰을 다시 가리킬 수 없다.


In [ ]:
Q = "한 문장으로 자기소개."

# Day 1
rc = client.chat.completions.create(
    model=API_MODEL,
    messages=[{"role": "user", "content": Q}],
)

# Day 3  https://developers.openai.com/api/reference/resources/responses/methods/create
rr = client.responses.create(
    model=API_MODEL,
    input=Q,
)

print("Day 1 답:", rc.choices[0].message.content)
print("Day 3 답:", rr.output_text)
print("Day 3 id :", rr.id)


def keys_of(obj):
    """응답 객체의 최상위 키를 줄 맞춰 찍는다 (39개라 한 줄로 보면 안 읽힌다)"""
    ks = sorted(obj.model_dump().keys())
    for i in range(0, len(ks), 6):
        print("   ", "  ".join(f"{k:22s}" for k in ks[i:i + 6]).rstrip())
    print(f"    → 모두 {len(ks)}개")


print()
print("[chat] 최상위 키")
keys_of(rc)
print()
print("[resp] 최상위 키")
keys_of(rr)


`keys_of`는 강사 코드다. Chat은 답만 담긴 봉투(키 약 9개), Responses는 내가 보낸 설정까지 되돌아오는 서버 저장 객체(키 약 39개).

공식 레퍼런스 — 어떤 인자를 `create`에 넣을 수 있는지:

https://developers.openai.com/api/reference/resources/responses/methods/create


---

## 흐름 2. 멀티턴 — 맥락을 누가 들고 가나

같은 네 문장. Day 1은 배열이 눈에 보이게 자란다. Day 3는 내가 보내는 글자는 짧은데, 청구 토큰은 는다.

```
Day 1                         Day 3
─────                         ─────
messages = [system]           prev = None
user 를 append                input = 이번 한 줄만
create(messages=통째)         create(input, previous_response_id=prev)
assistant 를 append           prev = r.id
다시 처음부터 부친다           서버가 앞 대화를 다시 투입
```


In [ ]:
turns = [
    "내 이름은 금이야. 취미는 등산이고 서울에 살아.",
    "내 취미가 뭐랬지?",
    "내가 어디 산다고 했지?",
    "내 이름은?",
]

print(f"{'턴':>2} | {'방식':^22} | {'보낸 글자':>8} | {'청구 입력':>8} | {'받은 토큰':>8}")
print("-" * 64)

# --- Day 1: 내가 배열을 든다 ---
messages = [{"role": "system", "content": "두 문장 이내로 답해."}]
for i, q in enumerate(turns, 1):
    messages.append({"role": "user", "content": q})
    r = client.chat.completions.create(model=API_MODEL, messages=messages)
    messages.append({"role": "assistant", "content": r.choices[0].message.content})
    print(
        f"{i:>2} | {'Day1 messages 통째':<22} | {len(q):>8} | "
        f"{r.usage.prompt_tokens:>8} | {r.usage.completion_tokens:>8}"
    )

print()

# --- Day 3: 서버가 들고, 나는 번호표만 ---
prev = None
for i, q in enumerate(turns, 1):
    kwargs = dict(model=API_MODEL, input=q)
    if prev:
        kwargs["previous_response_id"] = prev
    r = client.responses.create(**kwargs)
    prev = r.id
    print(
        f"{i:>2} | {'Day3 previous_response_id':<22} | {len(q):>8} | "
        f"{r.usage.input_tokens:>8} | {r.usage.output_tokens:>8}"
    )

print()
print("마지막 답 (Day 3):", r.output_text)
print("마지막 id :", r.id)


표에서 볼 것:

- **보낸 글자**는 Day 3가 더 짧다. 이번 한 줄만 보낸다.
- **청구 입력 토큰**은 둘 다 턴이 늘수록 커진다.

`previous_response_id` 가 줄이는 것은 보내는 양이지 요금이 아니다. 앞 대화는 사라진 게 아니라 서버로 옮겨 갔고, 서버가 매 턴 모델에 다시 넣는다. Day 1은 배열이 보여서 위험했고, Day 3는 안 보여서 더 위험하다.


### 흐름 2 보강. 남기지 않으면 번호표가 소용없다

`store=False` 는 모델에게 부탁하는 문장이 아니다. 저장 스위치다. 끈 채로 받은 `id` 를 `previous_response_id` 에 넣으면 400이 난다. 서버 문장에 고칠 곳이 적혀 있다.


In [ ]:
from openai import BadRequestError, NotFoundError

unsaved = client.responses.create(
    model=API_MODEL,
    input="짧게 인사만 해.",
    store=False,
)
print("store=False 로 받은 id:", unsaved.id)

try:
    client.responses.create(
        model=API_MODEL,
        input="방금 내가 뭘 말했지?",
        previous_response_id=unsaved.id,
    )
except BadRequestError as e:
    print("이어 붙이기 400:")
    print(str(e)[:300])

# 저장한 객체는 다시 꺼낼 수 있고, 지우면 404.
saved = client.responses.create(model=API_MODEL, input="한 단어로 안녕.")
got = client.responses.retrieve(saved.id)
print("retrieve 답:", got.output_text)
client.responses.delete(saved.id)
try:
    client.responses.retrieve(saved.id)
except NotFoundError as e:
    print("delete 뒤 retrieve 404:", str(e)[:200])


---

## 흐름 3. 구조화 출력 — 약속의 자리만 이동

클래스는 어제 그것이다. 바뀐 것은 넘기는 칸 이름과, 객체를 꺼내는 경로다.

```
같은 ReviewAnalysis 클래스
        │
        ├─ Day 2  parse(..., response_format=ReviewAnalysis)
        │         → r.choices[0].message.parsed
        └─ Day 3  parse(..., text_format=ReviewAnalysis)
                  → r.output_parsed
```


In [ ]:
class ReviewAnalysis(BaseModel):
    sentiment: str
    rating: int
    summary: str

rp = client.responses.parse(
    model=API_MODEL,
    input="이 리뷰를 분석해 줘: 배송은 느렸지만 물건은 기대 이상. 또 살 듯.",
    text_format=ReviewAnalysis,          # 어제는 response_format= 이었다. 이름만 다르다
)
print(rp.output_parsed)
print("타입:", type(rp.output_parsed), "| rating:", rp.output_parsed.rating)


---

## 흐름 4. 내 함수 도구 — 왕복은 사라지지 않았다

강사 코드(10:28 · 10:36). 함수는 내가 실행한다. 설명서는 한 겹 얕다. 재투입은 결과 조각 하나.

```
① tools 설명서
② output 에 function_call  (output_text 는 빈 문자열)
③ args = json.loads(fc.arguments) → get_weather(**args)   ← 실시간은 이 줄
④ input=[{type: function_call_output, call_id, output}]
⑤ r2.output_text
```

`additionalProperties: False` 는 Day 2 함정 B에서 배운 그 줄이다.


In [ ]:
def get_weather(city: str, latitude: float, longitude: float) -> str:
    """그 좌표의 지금 날씨를 open-meteo 에서 가져온다 (키 불필요)"""
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
           f"&current=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto")
    with urllib.request.urlopen(url, timeout=10) as resp:
        c = json.load(resp)["current"]
    return (f"{city} 기온 {c['temperature_2m']}도, 습도 {c['relative_humidity_2m']}%, "
            f"바람 {c['wind_speed_10m']}m/s")

print(get_weather("서울", 37.5665, 126.9780))

tools = [{
    "type": "function",
    "name": "get_weather",                      # 어제는 function 안에 있었다
    "description": "특정 도시의 현재 날씨(기온·습도·바람)를 가져온다. 실시간 정보가 필요할 때 쓴다.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
            "latitude": {"type": "number", "description": "위도"},
            "longitude": {"type": "number", "description": "경도"},
        },
        "required": ["city", "latitude", "longitude"],
        "additionalProperties": False,          # 어제 함정 B 에서 배운 그 줄
    },
}]


In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="서울 지금 몇 도야?",
    tools=tools,
)
print("output 아이템 :", [o.type for o in r.output])
print("output_text   :", repr(r.output_text))

fc = next(o for o in r.output if o.type == "function_call")
print("부르라는 함수 :", fc.name)
print("인자 문자열   :", fc.arguments)


In [ ]:
args = json.loads(fc.arguments)          # 모델이 준 인자는 '문자열'이다. 파이썬 값으로 되돌린다
result = get_weather(**args)             # ← 여기서 실제로 실행된다. 실시간 날씨는 이 줄에서 나온다
print("내 코드의 실행 결과 :", result)

r2 = client.responses.create(
    model=API_MODEL,
    previous_response_id=r.id,           # 앞 대화는 서버가 들고 있다
    tools=tools,
    input=[{                             # 새로 보내는 것은 '결과 조각' 하나뿐
        "type": "function_call_output",  # 어제의 {"role": "tool", ...} 자리
        "call_id": fc.call_id,           # 어느 요청에 대한 답인지 짝을 짓는다
        "output": result,
    }],
)
print("최종 답 :", r2.output_text)


Day 2는 요청서까지 배열에 다시 넣는다. Day 3는 서버가 요청서를 들고 있어서 **결과 조각만** 보낸다.

output_text 가 빈 문자열이면 실패가 아니다. [o.type for o in r.output] 에 unction_call 이 있는지 본다.
인자는 문자열이라 json.loads(fc.arguments) 로 파이썬 값으로 되돌린 다음 get_weather(**args) 한다.


---

## 흐름 4.5. 프롬프트 캐시 — 앞부분이 같으면 다시 안 읽는다

강사 코드(11:43). `[me]`는 이번 실행만의 앞부분을 만들어, 1회차는 캐시 0에서 시작하게 한다.

규정 200줄을 **앞에** 고정하고, 질문만 **뒤에** 바꾼다. 1,024토큰이 넘고 앞부분이 한 글자도 같아야 2회차에 `cached_tokens`가 오른다.


In [ ]:
me = uuid.uuid4().hex[:8]

rules = (f"[{me}] 당신은 사내 규정 안내 도우미입니다. 아래 규정을 근거로만 답합니다.\n"
      + "\n".join(f"규정 {i}: 사원은 항목 {i} 에 대해 담당 부서에 문의한다. 처리 기한은 {i % 7 + 1}일이다."
                  for i in range(1, 200)))

print("규정 글자 수:", len(rules), "| me:", me)

for i, q in enumerate(["규정 12 의 처리 기한은?", "규정 30 의 처리 기한은?"], 1):
    t0 = time.time()
    r_cache = client.responses.create(model=API_MODEL, instructions=rules, input=q)
    cached = r_cache.usage.input_tokens_details.cached_tokens
    print(
        f"[{i}회] {time.time() - t0:.1f}초 · "
        f"입력 {r_cache.usage.input_tokens}토큰 중 캐시 {cached}토큰 · "
        f"{r_cache.output_text[:60]}"
    )


1회 `cached_tokens` ≈ 0, 2회에 큰 수가 나오면 걸린 것이다. 규정은 `instructions=`(앞), 질문은 `input=`(뒤). `rules`를 루프 안에서 다시 만들면 `[me]`가 바뀌어 안 걸린다.


---

## 흐름 5. 내장 도구 — 왕복이 내 코드에서 사라진다

`web_search` 는 설명서도, `json.loads` 도, 재투입 루프도 없다. 한 줄이다.

```
Day 2·3 내 함수     ①설명서 → ②요청서 → ③내 컴퓨터 실행 → ④재투입 → ⑤말
Day 3  web_search   ①tools=[{type:web_search}] → (서버 안에서 ②③④) → ⑤말
```

편해진 만큼 검색 결과가 입력 토큰으로 들어온다. 공짜가 아니다.

아래 셀은 토큰이 꽤 나간다. 처음 복습이면 코드를 읽기만 하고, 필요할 때 `RUN_WEB_SEARCH = True` 로 바꾼다.


In [ ]:
RUN_WEB_SEARCH = False  # True 로 바꾸면 실제 검색. 입력 토큰이 수천까지 간다.

if not RUN_WEB_SEARCH:
    print("건너뜀. 흐름만 보면 된다.")
    print("켰을 때 볼 자리: [o.type for o in r.output] 에 web_search_call 이 있는가")
    print("               o.action.query  = 실제 검색어")
    print("               usage.input_tokens 가 크게 늘었는가")
else:
    rw = client.responses.create(
        model=API_MODEL,
        input="오늘 서울 날씨 한 줄로.",
        tools=[{"type": "web_search"}],
    )
    print("아이템:", [o.type for o in rw.output])
    print("답:", rw.output_text)
    for o in rw.output:
        if o.type == "web_search_call":
            print("검색함 · 상태:", o.status)
            print("검색어:", getattr(o.action, "query", None))
    print("입력 토큰:", rw.usage.input_tokens)


도구를 **등록**하는 것과 모델이 그것을 **쓰는** 것은 별개다. 판단은 모델이 한다. 강제로 쓰게 하려면 `tool_choice` 다.

```python
tool_choice={"type": "code_interpreter"}   # 무조건 이 도구
tool_choice="auto"                         # 기본. 모델이 고른다
tool_choice="none"                         # 도구를 실어도 안 씀
```


---

## 흐름 6. 답이 오는 동안 — 기다리느냐 흘리느냐

Day 1 앱은 `create` 가 끝날 때까지 화면이 멈춘다. Day 3 는 사건 이름표를 보고 글자 조각만 고른다.

```
Day 1  create(...)  ──(전부 생성)──►  한 번에 content
Day 3  with stream:
         event.type == "response.output_text.delta"  조각마다 덧붙임
         final = stream.get_final_response()         상자 완성본
```

스트리밍은 전체가 빨라지는 기술이 아니다. **첫 글자까지의 시간**을 바꾸는 기술이다.


In [ ]:
Q = "파이썬 리스트를 두 문장으로 설명해."

t0 = time.time()
waited = client.responses.create(model=API_MODEL, input=Q)
waited_sec = time.time() - t0
print(f"Day 3 create (기다림) {waited_sec:.2f}초 뒤에 한 번에:")
print(waited.output_text)
print()

t0 = time.time()
first_at = None
chunks = []
print("Day 3 stream (흘림)   조각이 오는 대로:")
with client.responses.stream(model=API_MODEL, input=Q) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            if first_at is None:
                first_at = time.time() - t0
            chunks.append(event.delta)
            print(event.delta, end="", flush=True)
    final = stream.get_final_response()
print()
print(f"첫 글자까지 {first_at:.2f}초 · 전체 {time.time() - t0:.2f}초 · 조각 {len(chunks)}개")
print("최종 id:", final.id)


브라우저 챗봇(`stream_app.py`)의 흐름은 터미널과 같다. 그릇만 `st.empty()` 다.

```
placeholder = st.empty()
answer = ""
with client.responses.stream(
    model=API_MODEL,
    input=prompt,                                 # 이번 말만
    previous_response_id=st.session_state.prev_id,
    text={"verbosity": length},                   # 라디오를 이 줄보다 위에서 읽는다
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            answer += event.delta
            placeholder.markdown(answer + "▌")
    final = stream.get_final_response()
st.session_state.prev_id = final.id
```

Day 1 앱에서 사라진 것: `session_state.messages` 배열, 답을 다 기다리던 시간. 남은 것: `prev_id` 한 줄.


---

## 흐름 7. 문서 여는 법 — llms.txt

강사 코드(14:16). 가이드 목차를 받아, 내 문제에 맞는 줄을 고른다. 레퍼런스를 처음부터 읽지 않는다.

- 가이드: 어떻게 하나 — https://developers.openai.com/api/docs/llms.txt
- 오늘 쓴 create 인자 사전: https://developers.openai.com/api/reference/resources/responses/methods/create


In [ ]:
with urllib.request.urlopen("https://developers.openai.com/api/docs/llms.txt", timeout=20) as resp:
    toc = resp.read().decode("utf-8")

print("목차 길이 :", len(toc), "자")
print("가이드 줄 :", sum(1 for line in toc.splitlines() if "/guides/" in line), "개")
for line in toc.splitlines():
    if "prompt-caching" in line or "rate-limits" in line or "your-data" in line:
        print(" ", line.strip()[:110])


---

## 한 장. 이름만 바뀐 것과, 흐름이 바뀐 것

| 자리 | Day 1·2 Chat Completions | Day 3 Responses |
|---|---|---|
| 부르기 | `chat.completions.create` | `responses.create` |
| 입력 | `messages=[{role, content}]` | `input="…"` 또는 아이템 목록 |
| 시스템 | `{"role":"system"}` | `instructions="…"` |
| 답 | `choices[0].message.content` | `output_text` |
| 멈춘 이유 | `finish_reason` | `status` + `incomplete_details.reason` |
| 길이 제한 | `max_completion_tokens` | `max_output_tokens` |
| 대화 잇기 | 배열 재전송 | `previous_response_id` |
| 구조화 | `parse(response_format=)` → `.parsed` | `parse(text_format=)` → `.output_parsed` |
| 도구 설명서 | `{type, function:{name,…}}` | `{type, name,…}` |
| 도구 재투입 | `{role:tool, tool_call_id}` | `{type:function_call_output, call_id}` |
| 상자 | 키 ~9개, retrieve 없음 | 키 ~39개, retrieve · delete |

외울 표가 아니다. 흐름이 갈리는 곳은 세 줄이다.

1. **결과냐 리소스냐** — Chat은 봉투. Responses는 서버에 남는 객체.
2. **칸이냐 목록이냐** — `message`의 고정 칸 vs `output[]`의 타입 아이템.
3. **입력도 목록** — `input="문자열"`은 줄임말. 도구 결과는 그 목록의 조각 하나.

그리고 **안 갈린 곳**: 내 함수를 실행하는 사람. 어제나 오늘이나 나다.


In [ ]:
print("이 노트북에서 붙잡은 상자")
print("[chat] 키", len(rc.model_dump()), "/ [resp] 키", len(rr.model_dump()))
print("Day 3 첫 호출 id 앞 20자:", rr.id[:20])


## 다음에 혼자 확인할 때

1. 답을 보기 전에, 오늘 호출이 **어느 흐름의 몇 번째 상자**인지 말한다.
2. `output_text` 가 비면 `print([o.type for o in r.output])` 부터 한다.
3. 토큰이 이상하면 보낸 글자가 아니라 **청구 입력**을 본다.
4. AI가 준 코드에 `chat.completions` 가 있으면 어제 것이다. 오늘 흐름으로 다시 짠다.

노트: [`내용정리.md`](./내용정리.md) · 문서 입구: https://developers.openai.com/api/docs/llms.txt


---

# 원본 수록 — `01_response_api.ipynb`

첫 호출 · 기억 · store · 키 비교 · 구조화 · 날씨 왕복 · 웹검색 · 코드 해석기


In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 자기소개 해봐. 한 문장으로."  # messages -> input
)

r.output_text   # <- choices[0].massage.content 대신에 쉽게 접근

In [ ]:
print(r.output_text, r.id) # 다른 점
print(r.usage.input_tokens, r.usage.output_tokens)  # usage는 token 사용은 같다.

# 기억

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 내 이름은 길동이야."
)
print(r.output_text)
r = client.responses.create(
    model=API_MODEL,
    input="내 이름이 뭐라고 했지?"
)
print(r.output_text)
# 각 API 간은 무상태성이라, 상태가 이어지지 않고, 기억 X

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="안녕, 내 이름은 길동이야."
)
print(r.output_text)
r = client.responses.create(
    model=API_MODEL,
    input="내 이름이 뭐라고 했지?",
    previous_response_id=r.id    # 이전 응답 ID를 매개변수로 주면 => 기억
)
print(r.output_text)

In [ ]:
prev = None
for i, q in enumerate(["내 이름은 금이야. 취미는 등산이고 서울에 살아.",
                       "내 취미가 뭐랬지?", "내가 어디 산다고 했지?", "내 이름은?"], 1):
    rr = client.responses.create(model=API_MODEL, input=q, previous_response_id=prev)
    prev = rr.id
    print(f"{i}번째턴, 입력토큰 : {rr.usage.input_tokens}, 출력토큰 : {rr.usage.output_tokens}, 내용 : {rr.output_text} ")
# API 메시지 배열이 안보여도 멀티턴 입력 토큰 누적량은 같다.

- 차이점 :
  - chat : 배열을 내가 들고 있고 -> (리스트를 내가 조작 가능하다)
    - 맥락을 다른 LLM 제공 업체로 옮길 수 있음
  - responses : 배열을 서버가 들고 있다. -> (리스트 조작이 불가능)
    - 맥락을 내가 소유하고 있지 않다.

In [ ]:
# 맥락을 서버에 저장하지 않고 싶을 경우
r = client.responses.create(
    model=API_MODEL,
    input="안녕, hello",
    store=False, # 서버에 안 남는다.
)
print(r.id)
print(r.output_text)

In [ ]:
try:
    r = client.responses.create(
        model=API_MODEL,
        input="내가 아까 뭐라고 했어?",
        previous_response_id=r.id  #    store=False 했던 id는 아예 사용 안됨
    )
except Exception as e:
    print(str(e))
# Error code: 400 - {'error': {'message': "Previous response with id .. not found

## 시스템 프롬프트는?

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    instructions="나는 초등학교 선생님이야. 쉽게 설명해줘",
    input="토큰이 뭐니?",
)
print(r.id)
print(r.output_text)

## 두 API 응답 객체 비교

In [ ]:
rc = client.chat.completions.create(
    model=API_MODEL,
    messages=[{"role":"user","content":"한 단어로 인사해"}]
)

rr = client.responses.create(
    model=API_MODEL,
    input="한 단어로 인사해",
)

In [ ]:
# API 응답 객체 구조와 키 수도 많이 다름.
print(rr)
print(rc)
print(len(rr.model_dump().keys()), rr.model_dump().keys()) # 39
print(len(rc.model_dump().keys()), rc.model_dump().keys()) # 9

In [ ]:
def keys_of(obj):
    """응답 객체의 최상위 키를 줄 맞춰 찍는다 (39개라 한 줄로 보면 안 읽힌다)"""
    ks = sorted(obj.model_dump().keys())
    for i in range(0, len(ks), 6):
        print("   ", "  ".join(f"{k:22s}" for k in ks[i:i + 6]).rstrip())
    print(f"    → 모두 {len(ks)}개")

print("[chat] 최상위 키")
keys_of(rc)
print()
print("[resp] 최상위 키")
keys_of(rr)
# responses api 응답 객체는 서버에 저장된 객체

In [ ]:
# 1. C
rr = client.responses.create(
    model=API_MODEL,
    input="안녕 내 이름은 장원이야.",
)
# 2. R
obj = client.responses.retrieve(rr.id)  # 이전 요청-응답 객체를 검색

In [ ]:
obj.output_text, obj.usage.input_tokens

In [ ]:
# 3. DELETE
client.responses.delete(rr.id)   # 서버에서 응답 객체를 지우기.

In [ ]:
try:
    obj = client.responses.retrieve(rr.id) # 삭제된 객체는 서버에서 조회 불가
except Exception as e:
    print(e)

In [ ]:
rc.choices[0].model_dump()  # chat
rr.output # responses # 답이 담기는 곳의 그릇도 다 다르다.

In [ ]:
rr = client.responses.create(
    model=API_MODEL,
    input="국제 관계에 대해서 설명해줘",
    max_output_tokens=20  # 최대토큰, 파라미터 명이 다름.
)
print(rr.output_text)  # 출력이 안되고
print(rr.status)       # 안된 상태
print(rr.incomplete_details)  # 안된 이유에 대해 자세히

In [ ]:
# messages list를 입력하듯이 입력하는 방법 => 가능
rr = client.responses.create(
    model=API_MODEL,
    input=[{"role":"developer", "content":"너는 한 단어로만 대답한다."},
           {"role":"user", "content":"브라질의 수도는?"}],
)
print(rr.output_text)  # 출력이 안되고

## 구조화된 출력

In [ ]:
# 구조화된 출력 등 그대로 사용되지만, key, 매개변수 명 주의
from pydantic import BaseModel

class ReviewAnalysis(BaseModel):
    sentiment: str
    rating: int
    summary: str

rp = client.responses.parse(
    model=API_MODEL,
    input="이 리뷰를 분석해 줘: 배송은 느렸지만 물건은 기대 이상. 또 살 듯.",
    text_format=ReviewAnalysis,          # 어제는 response_format= 이었다. 이름만 다르다
)
print(rp.output_parsed)

## 툴 콜링

In [ ]:
import json
import urllib


def get_weather(city: str, latitude: float, longitude: float) -> str:
    """그 좌표의 지금 날씨를 open-meteo 에서 가져온다 (키 불필요)"""
    url = (f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
           f"&current=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto")
    with urllib.request.urlopen(url, timeout=10) as resp:
        c = json.load(resp)["current"]
    return (f"{city} 기온 {c['temperature_2m']}도, 습도 {c['relative_humidity_2m']}%, "
            f"바람 {c['wind_speed_10m']}m/s")

print(get_weather("서울", 37.5665, 126.9780))

In [ ]:
tools = [{
    "type": "function",
    "name": "get_weather",                      # 어제는 function 안에 있었다
    "description": "특정 도시의 현재 날씨(기온·습도·바람)를 가져온다. 실시간 정보가 필요할 때 쓴다.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
            "latitude": {"type": "number", "description": "위도"},
            "longitude": {"type": "number", "description": "경도"},
        },
        "required": ["city", "latitude", "longitude"],
        "additionalProperties": False,          # 어제 함정 B 에서 배운 그 줄
    },
}]

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
    tools=tools # <-
)

In [ ]:
r.output_text
r.output[0] # 추론 아이템
r.output[1] # 함수 호출

In [ ]:
r.output[1].arguments, r.output[1].name
args = json.loads(r.output[1].arguments)
result = get_weather(**args)
print(result)

In [ ]:
fc = r.output[1]   # 이전 출력에서 펑션 콜 객체 꺼내기
args = json.loads(fc.arguments)          # 모델이 준 인자는 '문자열'이다. 파이썬 값으로 되돌린다
result = get_weather(**args)             # ← 여기서 실제로 실행된다. 실시간 날씨는 이 줄에서 나온다
print("내 코드의 실행 결과 :", result)

r2 = client.responses.create(
    model=API_MODEL,
    previous_response_id=r.id,           # 앞 대화는 서버가 들고 있다
    tools=tools,
    input=[{                             # 새로 보내는 것은 '결과 조각' 하나뿐
        "type": "function_call_output",  # 어제의 {"role": "tool", ...} 자리
        "call_id": fc.call_id,           # 어느 요청에 대한 답인지 짝을 짓는다
        "output": result,
    }],
)
print("최종 답 :", r2.output_text)

In [ ]:
r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
)
print(r.output_text)

r = client.responses.create(
    model=API_MODEL,
    input="지금 부산 습도 얼마니?",
    tools=[{"type": "web_search"}]  # OpenAI 내장 웹 검색 도구
)
print(r.output_text)

In [ ]:
r.output

In [ ]:
# 웹 검색 도구 과정이 포함되어있고, 웹 검색 결과를 통째로 입력값으로 받기 때문에
# inputs 토큰이 많이 늘어난다.
r.usage.input_tokens, r.usage.output_tokens, r.usage.total_tokens

In [ ]:
# 코드 내장 도구
Q = "49029865 * 3212934 를 계산해서 숫자만 반환해줘."
ANSWER = 49029865 * 3212934

r = client.responses.create(model=API_MODEL,
                        input=Q,
                        tools=[{"type":"code_interpreter",
                                "container":{"type": "auto"}}]
                    )
print(r.output_text)
print(ANSWER)

In [ ]:
r.output  # 3단계의 추론 -> 도구 -> 출력

In [ ]:
r.usage.input_tokens, r.usage.output_tokens, r.usage.total_tokens

---

# 원본 수록 — `02_stream.ipynb`

create 시간 · 사건 전부 · 타입 개수 · delta · 첫 글자 시각


## 한번에 전체 응답을 받는데 걸리는 시간 계산

In [ ]:
import time
t0 = time.time()
q = "LLM과 트랜스포머에 대해서 5 문장으로 설명해줘."
r = client.responses.create(model=API_MODEL, input=q)
print(f"걸린 시간 :  {time.time() - t0}")
print(r.output_text)

stream

In [ ]:
with client.responses.stream(model=API_MODEL, input=q) as stream:
    for event in stream:
        print(event) # 서버에서 스트리밍 관리자가 답변 조각(Event)을 보내줌.

In [ ]:
# 타입을 세 봄
count = {}
with client.responses.stream(model=API_MODEL, input=q) as stream:
    for event in stream:
        count[event.type] = count.get(event.type, 0) + 1

for k, v in count.items():
    print(f"{k:30s}  {v}")

In [ ]:
with client.responses.stream(model=API_MODEL, input=q) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

In [ ]:
t0 = time.time()
first_text = None
with client.responses.stream(model=API_MODEL, input=q) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            if first_text is None:
                first_text = time.time() - t0
            print(event.delta, end="", flush=True)
print(f"\n첫 글자가 나올때까지 걸린시간 : {first_text}")

---

# 원본 수록 — `03_ops.ipynb`

재시도 · 캐시 · verbosity · reasoning · temperature · 방 · llms.txt


In [ ]:
client.max_retries  # 클라이언트가 기본적으로 재시도하는 횟수 2

In [ ]:
client.timeout # 요청 후 기다림의 시간을 최대 10분 600초로 잡는다.
# connect 5.0 5초 동안 연결 안되면 종료.

In [ ]:
slow_client = OpenAI(max_retries=0, timeout=10.0)  # 클라이언트 기본 옵션 변경 가능

## 프롬프트 캐시

In [ ]:
import uuid

me = uuid.uuid4().hex[:8]

rules = (f"[{me}] 당신은 사내 규정 안내 도우미입니다. 아래 규정을 근거로만 답합니다.\n"
      + "\n".join(f"규정 {i}: 사원은 항목 {i} 에 대해 담당 부서에 문의한다. 처리 기한은 {i % 7 + 1}일이다."
                  for i in range(1, 200)))
len(rules)

In [ ]:
rules

In [ ]:
r = client.responses.create(model=API_MODEL, 
                        instructions=rules,
                        input="규정 12의 처리기한은 몇 일?")
print(r.output_text)
print(f"입력 토큰 : {r.usage.input_tokens}")
print(f"캐시 토큰 : {r.usage.input_tokens_details.cached_tokens}")

In [ ]:
r = client.responses.create(model=API_MODEL, 
                        instructions=rules,
                        input="규정 30의 처리기한은 몇 일?")
print(r.output_text)
print(f"입력 토큰 : {r.usage.input_tokens}")
print(f"캐시 토큰 : {r.usage.input_tokens_details.cached_tokens}")

In [ ]:
r.usage.input_tokens_details

- 완전히 동일한 입력이 반복되는 경우(지시문, 시스템 프롬프트)
- 이후에 입력이 될 떄 입력 토큰이 캐싱된다. 가격이 0.1배
- 고정된 긴 문맥 입력 + 짧은 query
- 그렇지만, 반복이 필요없는 경우는 반복 입력을 최적화하는 것이 요금 절약 방법
- 캐시를 잘 사용하면, 긴 입력도 절약 가능.
- 가능하면 반복되는 영역을 앞 부분에 둘 것.

## 매개변수 살펴보기

In [ ]:
import inspect
params = inspect.signature(client.responses.create).parameters
print(len(params), "개")

In [ ]:
# text
Q = "파이썬에 대해 설명해줘."
r1 = client.responses.create(model=API_MODEL,
                        input=Q,
                        text={"verbosity": "low"})
print(r1.output_text)

In [ ]:
r2 = client.responses.create(model=API_MODEL,
                        input=Q,
                        text={"verbosity": "high"})
print(r2.output_text)

In [ ]:
print(r1.usage.input_tokens, r1.usage.output_tokens)
print(r2.usage.input_tokens, r2.usage.output_tokens)

---
## 추론 파라미터 비교

In [ ]:
r3 = client.responses.create(model="o4-mini", input="19은 소수인가? 답만 말해줘.",
                        reasoning={"effort": "low"})

In [ ]:
print(r3.output_text)
print(r3.usage.input_tokens, r3.usage.output_tokens)
print(r3.usage.output_tokens_details) # 추론토큰 = 출력토큰 비용

In [ ]:
r4 = client.responses.create(model="o4-mini", input="19은 소수인가? 답만 말해줘.",
                        reasoning={"effort": "high"})
print(r4.output_text)
print(r4.usage.input_tokens, r4.usage.output_tokens)
print(r3.usage.output_tokens_details) # 추론토큰 = 출력토큰 비용

In [ ]:
# 매개변수 중에는 모델 마다 지원하는 것이 있고, 안되는 것이 있다.
try:
    r = client.responses.create(model=API_MODEL, input="안녕", temperature=0.1)
    # r = client.responses.create(model="gpt-5.4-nano", input="안녕", temperature=0.1)
    print(r.output_text)
except Exception as e:
    print(e)
# Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}

In [ ]:
# metadata : 요청에 설정한 태그
r = client.responses.create(model=API_MODEL, input="안녕",
                            metadata={"lesson": "8월 19일자 강의", "block":"param"})
print(r.output_text)
print(r.metadata)

In [ ]:
# prompt_cache_key : 해당 캐시를 묶는 태그
r = client.responses.create(model=API_MODEL, input="안녕",
                            prompt_cache_key="1111")  # key를 공유한 캐시 데이터를 공유함
print(r.output_text)


In [ ]:
# ... 사용자의 식별의 위한 변수
r = client.responses.create(model=API_MODEL, input="안녕",
                            safety_identifier="사용자 001번님")  # 사용자 식별용으로 씀.

In [ ]:
# 대화방 만들기
room = client.conversations.create()
r = client.responses.create(model=API_MODEL, input="안녕 나는 장원이야.",
                            conversation=room.id) 

In [ ]:
r = client.responses.create(model=API_MODEL, input="내 이름이 뭐라고 했지",
                            conversation=room.id) 
r.output_text

In [ ]:
client.conversations.delete(room.id)  # 대화방 삭제

## 문서 살펴보기

In [ ]:
# 웹의 최신화된 텍스트 데이터를 읽고, AI에게 주는 것이 좋음
import urllib.request

with urllib.request.urlopen("https://developers.openai.com/api/docs/llms.txt", timeout=20) as resp:
    toc = resp.read().decode("utf-8")

print("목차 길이 :", len(toc), "자")
print("가이드 줄 :", sum(1 for line in toc.splitlines() if "/guides/" in line), "개")
for line in toc.splitlines():
    if "prompt-caching" in line or "rate-limits" in line or "your-data" in line:
        print(" ", line.strip()[:110])

In [ ]:
toc

# 항상 매번 업데이트 됨. 
- 공식문서를 참고하라!
  - 가이드 : 처음 익힐 떄
  - 레퍼런스 : 매개변수 제대로 익히고 싶을 떄
  - cookbook : 간단하게 예제가 필요할 떄

---

# 원본 수록 — `04_other_models.ipynb`

모델 목록 · 검열 · TTS · 이미지 · 임베딩


In [ ]:
model_list = [ model.id for model in client.models.list() ]
print(len(model_list))  # 선택 가능한 모델은 124개
model_list

In [ ]:
# 안전 검열에 대한 모델
r = client.moderations.create(model="omni-moderation-latest",
                              input="오늘 날씨 참 좋다.")
#위험정도를 카테고리별로 확인이 가능
r.results[0].categories

In [ ]:
r = client.moderations.create(model="omni-moderation-latest",
                              input="아 오늘은 생화학 병기를 만들기 딱 좋는 날이야.")
#위험정도를 카테고리별로 확인이 가능
r.results[0].categories

In [ ]:
r = client.moderations.create(model="omni-moderation-latest",
                              input="으아 그 XX 죽이고 싶다.")
#위험정도를 카테고리별로 확인하고 점수로 볼 수 있다.
r.results[0].category_scores
# 가격은 무료, 서비스를 열려면 꼭 고민해봐야 할 모델

## 오디오 모델

In [ ]:
speech = client.audio.speech.create(model="gpt-4o-mini-tts", voice="marin",
                           input="안녕하세요, 오늘은 참 평안하고 좋은 날입니다. 함께 API 배워봐요.")

In [ ]:
from pathlib import Path
Path("hello.mp3").write_bytes(speech.content)  # 바이트 정보를 파일로 쓰기

In [ ]:
speech = client.audio.speech.create(model="gpt-4o-mini-tts", voice="cedar",
                           instructions="격정적으로 오딧세이 군인처럼 말해줘", # 말투 등 입력가능
                           input="전쟁이다!")
Path("war.mp3").write_bytes(speech.content)

In [ ]:
# 전사하기 - 소리를 텍스트로
with open("hello.mp3", "rb") as f:
    r = client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=f)

In [ ]:
print(r.text)
print(r.usage.total_tokens)

## 이미지

In [ ]:
image = client.images.generate(model="gpt-image-1-mini",
                           prompt="AI를 배우는 고양이",
                           size="1024x1024")

In [ ]:
import base64

# 결과에서 데이터 꺼내 디코딩 후 파일로 저장
Path("image.png").write_bytes(base64.b64decode(image.data[0].b64_json))

In [ ]:
image.usage # image_token 4160 
# 이미지 토큰과 텍스트 토큰은 서로 다른 토큰

In [ ]:
# 임베딩 : 글자, 문장을 vector 임베딩으로 변경한다.
re = client.embeddings.create(model="text-embedding-3-small",
                         input="강아지가 공을 물고 달린다.")


In [ ]:
vector = re.data[0].embedding

In [ ]:
len(vector)

In [ ]:
vector